# Combined Assignment Notebook
Includes SVM Deforestation Analysis PLUS Apriori/Frequent Itemset/Association Rule sections requested by evaluator.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from mlxtend.frequent_patterns import apriori, association_rules


In [ ]:

df = pd.read_csv('deforestation_dataset(1).csv')
df.head()


## SVM Preprocessing

In [ ]:

num_cols=df.select_dtypes(include='number').columns
cat_cols=df.select_dtypes(exclude='number').columns

df[num_cols]=df[num_cols].fillna(df[num_cols].median())

for c in cat_cols:
    df[c]=df[c].fillna(df[c].mode()[0])
    df[c]=LabelEncoder().fit_transform(df[c].astype(str))


In [ ]:

target='Forest_Loss_Area_km2'

X=df.drop(columns=[target])
y=df[target]

X=StandardScaler().fit_transform(X)

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

svm=SVR(kernel='linear')
svm.fit(X_train,y_train)

pred=svm.predict(X_test)

print('MAE',mean_absolute_error(y_test,pred))
print('MSE',mean_squared_error(y_test,pred))
print('RMSE',mean_squared_error(y_test,pred)**0.5)
print('R2',r2_score(y_test,pred))


In [ ]:

params={
    'kernel':['linear','rbf','poly'],
    'C':[0.1,1,10],
    'gamma':['scale','auto']
}

grid=GridSearchCV(SVR(),params,cv=5,scoring='r2')
grid.fit(X_train,y_train)

print('Best Params:',grid.best_params_)
print('Best Score:',grid.best_score_)

scores=cross_val_score(grid.best_estimator_,X,y,cv=5,scoring='r2')
print('CV Mean R2:',scores.mean())


## Frequent Itemsets & Apriori (Added for Evaluation Rubric)

In [ ]:

# Convert selected variables into categorical bins for Apriori

basket=pd.DataFrame()

basket['High_CO2']=pd.qcut(df['CO2_Emission_mt'],2,labels=['LowCO2','HighCO2'])
basket['High_Rainfall']=pd.qcut(df['Rainfall_mm'],2,labels=['LowRain','HighRain'])
basket['High_Population']=pd.qcut(df['Population'],2,labels=['LowPop','HighPop'])
basket['High_ForestLoss']=pd.qcut(df['Forest_Loss_Area_km2'],2,labels=['LowLoss','HighLoss'])

encoded=pd.get_dummies(basket)

encoded.head()


In [ ]:

frequent_itemsets=apriori(
    encoded,
    min_support=0.10,
    use_colnames=True
)

frequent_itemsets.sort_values(
    by='support',
    ascending=False
).head(20)


In [ ]:

rules=association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.5
)

rules[['antecedents',
       'consequents',
       'support',
       'confidence',
       'lift']].head(20)


In [ ]:

# Rule evaluation

print('Average Confidence:',rules['confidence'].mean())
print('Average Lift:',rules['lift'].mean())


In [ ]:

# Simple rule validation using folds

from sklearn.model_selection import KFold

kf=KFold(n_splits=5,shuffle=True,random_state=42)

supports=[]

for train_idx,test_idx in kf.split(encoded):
    train_data=encoded.iloc[train_idx]

    fi=apriori(train_data,min_support=0.10,use_colnames=True)

    if len(fi)>0:
        supports.append(fi['support'].mean())

print('Cross-validated Average Support:',np.mean(supports))


In [ ]:

# Frequent itemset visualization

top=frequent_itemsets.sort_values(
    'support',
    ascending=False
).head(10)

plt.figure(figsize=(8,5))
plt.bar(range(len(top)),top['support'])
plt.title('Top Frequent Itemsets')
plt.show()


In [ ]:

# Association rule scatter plot

plt.figure(figsize=(8,5))
plt.scatter(rules['support'],rules['confidence'])
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules')
plt.show()



## Final Report

### SVM Findings
- Predict deforestation using socioeconomic and environmental variables.
- Evaluate using MAE, MSE, RMSE and R².

### Apriori Findings
- Extract frequent itemsets.
- Generate association rules.
- Evaluate rules using support, confidence and lift.
- Validate using cross-validation.

### Recommendations
- Strengthen anti-deforestation policies.
- Reduce illegal lumbering.
- Increase protected forest areas.
- Monitor high CO2 and high forest-loss regions.
